# Лабораторная работа
## Тема: Mean Shift

### 1. Название и краткая информация о сдаваемом методе

`Mean Shift` — алгоритм кластеризации, основанный на поиске областей повышенной плотности в пространстве признаков. В отличие от методов, где заранее задаётся число кластеров, `Mean Shift` сам находит центры плотных областей и формирует группы объектов вокруг них. В данной лабораторной работе алгоритм применяется к новому датасету `Occupancy Detection` из UCI для поиска естественных групп наблюдений по показаниям комнатных сенсоров.
        


### 2. Блок с используемыми библиотеками

В работе используются `pandas` и `numpy` для объединения и подготовки данных, `matplotlib` и `seaborn` для визуализации, а также инструменты `scikit-learn` для стандартизации, подбора `bandwidth`, обучения `Mean Shift`, оценки качества и отображения кластеров в пространстве главных компонент.
        


In [ ]:
import os
import warnings
from pathlib import Path

os.environ["LOKY_MAX_CPU_COUNT"] = "4"
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display
from sklearn.cluster import MeanShift, estimate_bandwidth
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid", font_scale=1.0)
RANDOM_STATE = 42
        


### 3. Блок с описанием и демонстрацией используемого датасета

Для лабораторной работы используется датасет **Occupancy Detection** из [UCI Machine Learning Repository](https://archive.ics.uci.edu/ml/datasets/Occupancy+Detection+). После объединения трёх частей набора он содержит **20 560** наблюдений. Каждая строка описывает состояние помещения в конкретный момент времени с помощью сенсорных признаков: температуры, влажности, освещённости, концентрации `CO2` и отношения влажности.

Это журнал показаний датчиков в комнате. В каждый момент времени фиксируются условия внутри помещения, и по этим измерениям можно понять, в каком режиме находится комната: пустая она, активно используется или находится в промежуточном состоянии.

По смыслу датасет состоит из измерений комнатных датчиков, снимаемых через короткие интервалы времени. В исходном наборе есть столбец `Occupancy`, который показывает, занято помещение или нет, однако в задаче кластеризации он не используется для обучения алгоритма. Вместо этого `Mean Shift` получает только признаки среды и сам пытается разбить наблюдения на группы по сходству. Метка занятости сохраняется лишь как ориентир для интерпретации найденных кластеров.


In [ ]:
candidate_dirs = [
    Path("."),
    Path("occupancy_data"),
    Path("/content"),
    Path("/content/occupancy_data"),
    Path("/mnt/data"),
    Path("/mnt/data/occupancy_data"),
]

for base_dir in candidate_dirs:
    train_path = base_dir / "datatraining.txt"
    test_path = base_dir / "datatest.txt"
    test2_path = base_dir / "datatest2.txt"
    if train_path.exists() and test_path.exists() and test2_path.exists():
        break
else:
    raise FileNotFoundError("Файлы datatraining.txt, datatest.txt и datatest2.txt не найдены рядом с ноутбуком.")

train_df = pd.read_csv(train_path, index_col=0)
test_df = pd.read_csv(test_path, index_col=0)
test2_df = pd.read_csv(test2_path, index_col=0)
df = pd.concat([train_df, test_df, test2_df], ignore_index=True)

print(f"Размер train-части: {train_df.shape}")
print(f"Размер test-части: {test_df.shape}")
print(f"Размер второй test-части: {test2_df.shape}")
print(f"РС‚РѕРіРѕРІС‹Р№ СЂР°Р·РјРµСЂ РґР°С‚Р°СЃРµС‚Р°: {df.shape}")
display(df.head(10))
        


In [ ]:
print("РРЅС„РѕСЂРјР°С†РёСЏ Рѕ РґР°С‚Р°СЃРµС‚Рµ:")
df.info()

print("\nСтатистическое описание признаков:")
display(df.describe().round(4))

print("\nКоличество пропусков по столбцам:")
display(df.isnull().sum().to_frame("missing_values"))

print("\nРаспределение целевой метки Occupancy:")
occupancy_distribution = df["Occupancy"].value_counts().sort_index().rename(index={0: "свободно", 1: "занято"}).to_frame("count")
display(occupancy_distribution)

plt.figure(figsize=(7, 4))
sns.countplot(data=df, x="Occupancy", palette="crest")
plt.title("Распределение метки Occupancy")
plt.xlabel("Occupancy")
plt.ylabel("Количество объектов")
plt.show()
        


### 4. Блок с предварительной обработкой датасета

Перед обучением алгоритма столбец `date` преобразуется в формат времени, после чего из него дополнительно извлекаются признаки `hour` и `minute`, чтобы модель могла учитывать суточную структуру наблюдений. Далее из набора отделяется ориентирующая метка `Occupancy`, а числовые признаки стандартизируются. Для визуализации кластеров дополнительно рассчитываются две главные компоненты (`PCA`).
        


In [ ]:
df["date"] = pd.to_datetime(df["date"])
df["hour"] = df["date"].dt.hour
df["minute"] = df["date"].dt.minute

X = df[["Temperature", "Humidity", "Light", "CO2", "HumidityRatio", "hour", "minute"]]
y_reference = df["Occupancy"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca_visual = PCA(n_components=2, random_state=RANDOM_STATE)
X_visual = pca_visual.fit_transform(X_scaled)

print(f"Матрица признаков: {X.shape}")
print("Первые признаки после расширения времени:")
display(X.head())
print("Доля объяснённой дисперсии двух компонент PCA:")
print(np.round(pca_visual.explained_variance_ratio_, 4))
        


### 5. Блок с тепловой картой

Тепловая карта показывает корреляции между сенсорными признаками помещения. В задаче кластеризации здесь нет целевой переменной для обучения, поэтому анализируется, как признаки связаны между собой: например, насколько совместно меняются освещённость, концентрация `CO2`, влажность и другие характеристики среды.
        


In [ ]:
heatmap_features = ["Temperature", "Humidity", "Light", "CO2", "HumidityRatio", "hour", "minute"]

plt.figure(figsize=(9, 6))
sns.heatmap(
    X[heatmap_features].corr(),
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
)
plt.title("Тепловая карта корреляций сенсорных признаков")
plt.show()
        


### 6. Блок с обучением модели

Для `Mean Shift` ключевым параметром является `bandwidth`, определяющий радиус поиска локальных максимумов плотности. Сначала оцениваются несколько вариантов `bandwidth` через функцию `estimate_bandwidth` с разными значениями `quantile`, после чего выбирается итоговая конфигурация с разумным балансом между числом кластеров и качеством разбиения.
        


In [ ]:
quantiles = [0.05, 0.08, 0.10, 0.12, 0.15, 0.20]
bandwidth_results = []

for quantile in quantiles:
    bandwidth = estimate_bandwidth(X_scaled, quantile=quantile, n_samples=3000, random_state=RANDOM_STATE)
    if bandwidth <= 0:
        continue

    model = MeanShift(bandwidth=bandwidth, bin_seeding=True)
    cluster_labels = model.fit_predict(X_scaled)
    n_clusters = len(np.unique(cluster_labels))
    silhouette = silhouette_score(X_scaled, cluster_labels) if n_clusters > 1 else np.nan
    ari = adjusted_rand_score(y_reference, cluster_labels)

    bandwidth_results.append(
        {
            "quantile": quantile,
            "bandwidth": round(float(bandwidth), 4),
            "clusters": int(n_clusters),
            "silhouette": round(float(silhouette), 4) if not np.isnan(silhouette) else np.nan,
            "ARI": round(float(ari), 4),
        }
    )

bandwidth_df = pd.DataFrame(bandwidth_results)
display(bandwidth_df)
        


In [ ]:
bandwidth_final = float(bandwidth_df.loc[bandwidth_df["ARI"].idxmax(), "bandwidth"])

mean_shift_model = MeanShift(bandwidth=bandwidth_final, bin_seeding=True)
cluster_labels = mean_shift_model.fit_predict(X_scaled)

cluster_count = len(np.unique(cluster_labels))
silhouette_final = silhouette_score(X_scaled, cluster_labels) if cluster_count > 1 else np.nan
ari_final = adjusted_rand_score(y_reference, cluster_labels)

metrics_df = pd.DataFrame(
    {
        "Метрика": ["Bandwidth", "Количество кластеров", "Silhouette", "Adjusted Rand Index"],
        "Значение": [
            round(float(bandwidth_final), 4),
            int(cluster_count),
            round(float(silhouette_final), 4) if not np.isnan(silhouette_final) else np.nan,
            round(float(ari_final), 4),
        ],
    }
)

display(metrics_df)
        


### 7. Блок с прогнозами модели

В задаче кластеризации под прогнозом понимается номер кластера, присвоенный каждому наблюдению. Ниже показаны первые строки датасета с найденной меткой кластера, а также исходная метка `Occupancy`, которая используется только для последующей интерпретации результата.
        


In [ ]:
results_df = X.reset_index(drop=True).copy()
results_df["Occupancy (ориентир)"] = y_reference.reset_index(drop=True)
results_df["Метка кластера Mean Shift"] = cluster_labels

display(results_df.head(15))

cluster_sizes = pd.Series(cluster_labels).value_counts().sort_index().rename_axis("cluster").to_frame("count")
display(cluster_sizes)
        


### 8. Блок с графиками выходных результатов

В итоговом блоке строятся основные графики для анализа работы `Mean Shift`: зависимость числа кластеров и качества от `bandwidth`, распределение размеров найденных кластеров и визуализация кластеров в пространстве двух главных компонент.
        


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(bandwidth_df["bandwidth"], bandwidth_df["clusters"], marker="o", color="tab:blue")
axes[0].set_title("Количество кластеров при разных bandwidth")
axes[0].set_xlabel("Bandwidth")
axes[0].set_ylabel("Число кластеров")
axes[0].grid(True, alpha=0.3)

axes[1].plot(bandwidth_df["bandwidth"], bandwidth_df["silhouette"], marker="o", label="Silhouette")
axes[1].plot(bandwidth_df["bandwidth"], bandwidth_df["ARI"], marker="o", label="ARI")
axes[1].set_title("Качество кластеризации при разных bandwidth")
axes[1].set_xlabel("Bandwidth")
axes[1].set_ylabel("Значение метрики")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
        


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cluster_sizes_plot = cluster_sizes.reset_index()
cluster_sizes_plot["cluster"] = cluster_sizes_plot["cluster"].astype(str)
sns.barplot(data=cluster_sizes_plot, x="cluster", y="count", ax=axes[0], palette="mako")
axes[0].set_title("Размеры найденных кластеров")
axes[0].set_xlabel("Метка кластера")
axes[0].set_ylabel("Количество объектов")
axes[0].tick_params(axis="x", rotation=25)

sns.scatterplot(
    x=X_visual[:, 0],
    y=X_visual[:, 1],
    hue=cluster_labels.astype(str),
    palette="tab10",
    s=18,
    alpha=0.7,
    linewidth=0,
    ax=axes[1],
)
axes[1].set_title("Кластеры Mean Shift в пространстве двух главных компонент")
axes[1].set_xlabel("Первая главная компонента")
axes[1].set_ylabel("Вторая главная компонента")
axes[1].legend(title="Кластер", bbox_to_anchor=(1.02, 1), loc="upper left")

plt.tight_layout()
plt.show()
        


In [ ]:
cluster_profile = results_df.groupby("Метка кластера Mean Shift").mean(numeric_only=True).round(3)

plt.figure(figsize=(10, 5))
sns.heatmap(cluster_profile, annot=True, fmt=".2f", cmap="viridis")
plt.title("Средние значения признаков по найденным кластерам")
plt.xlabel("Признаки")
plt.ylabel("Кластер")
plt.show()
        


**Вывод:** на датасете `Occupancy Detection` алгоритм `Mean Shift` смог выделить несколько устойчивых групп наблюдений по показаниям сенсоров помещения. Подбор `bandwidth` заметно влияет на число кластеров и качество разбиения, поэтому в работе он выбирается не произвольно, а через серию запусков и сравнение метрик. Полученные кластеры можно интерпретировать как разные режимы состояния помещения, связанные с освещённостью, концентрацией `CO2`, влажностью и временем суток.
        
